# 📊 SP500 Fundamental Data Downloader

ดึงข้อมูล Fundamental ของหุ้น S&P 500 จาก **yfinance**

| ข้อมูล | รายละเอียด |
|--------|------------|
| Income Statement | Revenue, EPS, Net Income, EBITDA |
| Balance Sheet | Assets, Debt, Equity, Cash |
| Cash Flow | Operating CF, FCF, CapEx |
| Key Ratios | P/E, P/B, ROE, Debt/Equity |

> แยกจาก daily OHLCV pipeline — รันรายเดือนหลัง earnings season

In [ ]:
# Install dependencies
!pip install -q yfinance pandas

In [ ]:
import yfinance as yf
import pandas as pd
import time
import json
from datetime import datetime, timezone
from pathlib import Path

print(f"yfinance version: {yf.__version__}")
print(f"Run time: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")

## ⚙️ Config

In [ ]:
# === CONFIG ===
# เปลี่ยนเป็น None เพื่อดึงทั้ง S&P 500
# หรือระบุ list เพื่อทดสอบ
TEST_TICKERS = ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN"]  # Set to None for full S&P 500

OUTPUT_DIR = Path("fundamentals")
OUTPUT_DIR.mkdir(exist_ok=True)

RATE_LIMIT_DELAY = 0.5  # seconds between tickers

## 📋 Get Ticker List

In [ ]:
if TEST_TICKERS:
    tickers = TEST_TICKERS
    print(f"Using test tickers: {tickers}")
else:
    tables = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies")
    sp500_df = tables[0]
    tickers = sp500_df["Symbol"].str.replace(".", "-", regex=False).tolist()
    print(f"Loaded {len(tickers)} S&P 500 tickers")

print(f"First 10: {tickers[:10]}")

## 🔽 Download Fundamental Data

In [ ]:
# Fields to extract
INCOME_FIELDS = [
    "Total Revenue", "Net Income", "Gross Profit", "Operating Income",
    "EBITDA", "Basic EPS", "Diluted EPS", "Total Expenses",
    "Cost Of Revenue", "Research And Development",
]

BALANCE_FIELDS = [
    "Total Assets", "Total Liabilities Net Minority Interest",
    "Stockholders Equity", "Total Debt", "Cash And Cash Equivalents",
    "Net Debt", "Current Assets", "Current Liabilities",
    "Long Term Debt", "Working Capital",
]

CASHFLOW_FIELDS = [
    "Operating Cash Flow", "Free Cash Flow", "Capital Expenditure",
    "Investing Cash Flow", "Financing Cash Flow",
    "Repurchase Of Capital Stock", "Cash Dividends Paid",
    "Issuance Of Debt", "Repayment Of Debt",
]

RATIO_KEYS = {
    "trailingPE": "Trailing P/E", "forwardPE": "Forward P/E",
    "priceToBook": "P/B", "pegRatio": "PEG Ratio",
    "returnOnEquity": "ROE", "returnOnAssets": "ROA",
    "debtToEquity": "Debt/Equity", "currentRatio": "Current Ratio",
    "profitMargins": "Profit Margin", "operatingMargins": "Operating Margin",
    "revenueGrowth": "Revenue Growth", "earningsGrowth": "Earnings Growth",
    "trailingEps": "Trailing EPS", "forwardEps": "Forward EPS",
    "marketCap": "Market Cap", "dividendYield": "Dividend Yield",
    "beta": "Beta", "sector": "Sector", "industry": "Industry",
}

def safe_extract(df, fields):
    if df is None or df.empty:
        return {}
    result = {}
    for field in fields:
        if field in df.index:
            row = df.loc[field]
            result[field] = {
                col.strftime("%Y-%m-%d") if hasattr(col, "strftime") else str(col): (
                    float(val) if pd.notna(val) else None
                )
                for col, val in row.items()
            }
    return result

print("Functions ready ✅")

In [ ]:
# === MAIN DOWNLOAD LOOP ===
all_data = []
failed_tickers = []

for i, ticker in enumerate(tickers, 1):
    print(f"[{i}/{len(tickers)}] {ticker}...", end=" ")
    try:
        t = yf.Ticker(ticker)
        
        data = {
            "ticker": ticker,
            "annual": {
                "income": safe_extract(t.income_stmt, INCOME_FIELDS),
                "balance": safe_extract(t.balance_sheet, BALANCE_FIELDS),
                "cashflow": safe_extract(t.cashflow, CASHFLOW_FIELDS),
            },
            "quarterly": {
                "income": safe_extract(t.quarterly_income_stmt, INCOME_FIELDS),
                "balance": safe_extract(t.quarterly_balance_sheet, BALANCE_FIELDS),
                "cashflow": safe_extract(t.quarterly_cashflow, CASHFLOW_FIELDS),
            },
            "ratios": {},
        }
        
        # Ratios
        try:
            info = t.info or {}
            data["ratios"] = {label: info[key] for key, label in RATIO_KEYS.items() if key in info and info[key] is not None}
        except:
            pass
        
        all_data.append(data)
        print("✅")
        
    except Exception as e:
        print(f"❌ {e}")
        failed_tickers.append(ticker)
    
    if i < len(tickers):
        time.sleep(RATE_LIMIT_DELAY)

print(f"\n{'='*50}")
print(f"Done: {len(all_data)} success, {len(failed_tickers)} failed")
if failed_tickers:
    print(f"Failed: {failed_tickers}")

## 💾 Save to CSV

In [ ]:
def build_statement_csv(all_data, statement_type, period):
    rows = []
    for data in all_data:
        ticker = data["ticker"]
        statements = data.get(period, {}).get(statement_type, {})
        for metric, date_values in statements.items():
            for date, value in date_values.items():
                rows.append({"Ticker": ticker, "Date": date, "Metric": metric, "Value": value})
    return pd.DataFrame(rows)

def build_ratios_csv(all_data):
    rows = []
    for data in all_data:
        row = {"Ticker": data["ticker"]}
        row.update(data.get("ratios", {}))
        rows.append(row)
    return pd.DataFrame(rows)

# Save all CSVs
files_saved = []
for stmt, label in [("income","income"),("balance","balance"),("cashflow","cashflow")]:
    for period in ["annual", "quarterly"]:
        df = build_statement_csv(all_data, stmt, period)
        if not df.empty:
            path = OUTPUT_DIR / f"{label}_{period}.csv"
            df.to_csv(path, index=False)
            files_saved.append((str(path), len(df)))
            print(f"✅ {path} — {len(df)} rows")

# Ratios
df_ratios = build_ratios_csv(all_data)
if not df_ratios.empty:
    path = OUTPUT_DIR / "ratios_current.csv"
    df_ratios.to_csv(path, index=False)
    files_saved.append((str(path), len(df_ratios)))
    print(f"✅ {path} — {len(df_ratios)} rows")

print(f"\nTotal files: {len(files_saved)}")

## 🔍 Quick Preview

In [ ]:
# Preview income statement
df_income = pd.read_csv(OUTPUT_DIR / "income_annual.csv")
print("=== Income Annual (sample) ===")
display(df_income.head(20))

print("\n=== Ratios (sample) ===")
df_ratios = pd.read_csv(OUTPUT_DIR / "ratios_current.csv")
display(df_ratios.head())

## ☁️ Upload to GitHub (Optional)

ถ้าต้องการ push ขึ้น repo จาก Colab โดยตรง

In [ ]:
# === OPTIONAL: Push to GitHub ===
# Uncomment and fill in your token

# GITHUB_TOKEN = "ghp_xxxxxxxxxxxx"  # <-- ใส่ token ที่นี่
# REPO = "jptrustlearning/sp500"
# BRANCH = "main"

# !git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git repo_clone
# !cp -r fundamentals/ repo_clone/fundamentals/
# %cd repo_clone
# !git add fundamentals/
# !git commit -m "📊 Fundamentals update: $(date -u +'%Y-%m-%d %H:%M UTC')"
# !git push
# %cd ..